# OFFSIDE — the factory

**This notebook is the only place a model runs.** It bakes a real football incident into a frozen, audited `IncidentBundle` fixture, then the web app reads that JSON offline. Nothing here runs at web runtime.

The bake is deterministic (temperature 0, fixed seed): re-running it on the same corpus produces a byte-identical fixture. Every step below is tested code in `offside_engine` — this notebook only wires the three IBM Granite models to it.

**Runtime:** set **Runtime → Change runtime type → T4 GPU** before running. The three Granite models fit comfortably in a T4.

| Step | What it does |
|------|--------------|
| 1 | Install Ollama + the engine package |
| 2 | Pull the three IBM Granite models (generator, embedder, Guardian) |
| 3 | Get the StatsBomb aggregate (license-safe, derived only) |
| 4 | Bake: assemble evidence → index → 4 lenses → Guardian gate → THE SPLIT → freeze |
| 5 | Inspect the derived SPLIT and the fixture |
| 6 | Commit the fixture back to the repo |

## 1 — Install Ollama and the engine

Ollama serves the Granite models locally inside the Colab VM. The engine is installed from the repo so the exact tested pipeline runs — no logic is redefined in this notebook.

In [ ]:
# Install the Ollama server (Linux) and start it in the background.
# zstd is required by the Ollama installer to extract its release archive, and is not
# preinstalled on Colab — install it first or the install silently produces no binary.
!apt-get -qq update && apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, os, urllib.request
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
_ollama = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Wait until the server actually answers, rather than sleeping a fixed time.
def _ollama_ready(timeout_s=60):
    for _ in range(timeout_s):
        try:
            urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
            return True
        except Exception:
            time.sleep(1)
    return False

assert _ollama_ready(), "ollama server did not come up — check the install output above"
print("ollama serving: ready")

In [ ]:
# Clone the repo (for the corpus + the engine source) and install the engine.
# Remove any previous clone first so a re-run always picks up the LATEST pushed code —
# `git clone` silently skips an existing directory, which would run stale engine logic.
REPO_URL = "https://github.com/vighriday/offside-june-2026.git"
!rm -rf /content/offside
!git clone --depth 1 {REPO_URL} /content/offside
%cd /content/offside/engine
!pip -q install -e .

# Confirm we are on the commit we think we are — paste this SHA if anything looks off.
!git -C /content/offside log --oneline -1

## 2 — Pull the three IBM Granite models

Three models, three jobs — all IBM Granite:

* **`granite3.3:8b`** — the generator. Constrained to a no-numbers JSON schema, so it is *structurally* unable to emit a fabricated percentage.
* **`granite-embedding:30m`** — embeds the evidence so each lens retrieves only its own.
* **`granite3-guardian:2b`** — the auditor. A second IBM model that checks the first's groundedness and demotes anything it cannot confirm.

In [ ]:
!ollama pull granite3.3:8b
!ollama pull granite-embedding:30m
!ollama pull granite3-guardian:2b
!ollama list

## 3 — The StatsBomb aggregate (license-safe)

StatsBomb Open Data is used under its User Agreement: events are pulled at build time, **only derived aggregates are kept**, raw rows are never written. The one signal the Tactical lens cites is that the hand-scored goal falls back to the feed's catch-all body-part label — a statement about the *data model only*.

In [ ]:
from offside_engine.statsbomb.pull_aggregates import pull_hand_of_god_aggregate
aggregate = pull_hand_of_god_aggregate()
print("anomaly present:", aggregate.anomaly_present)
print("hand-of-god body part:", aggregate.hand_of_god_body_part)
print("comparison goal body part:", aggregate.second_goal_body_part)
print(aggregate.attribution)

## 4 — Bake the incident

`run_bake` runs the whole tested pipeline: assemble the evidence pool from the corpus, build the per-lens LanceDB index, run the four lenses (each cite-or-die), audit each with Guardian, synthesise THE SPLIT, audit each cell with Guardian, **assert the derived SPLIT matches the documented thesis**, and freeze the fixture.

The thesis assertion is the honesty check: the synthesis never sees the expected answer, so a pass proves the rules *derived* the Hand of God signature rather than being told it. If the derived SPLIT drifts, the bake fails here rather than freeze a wrong fixture.

In [ ]:
from pathlib import Path
import subprocess

from offside_engine.bake.incident import HAND_OF_GOD
from offside_engine.bake.runner import run_bake

REPO = Path("/content/offside")
corpus_sha = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"]).decode().strip()

result = run_bake(
    HAND_OF_GOD,
    framing_yaml=REPO / "corpus" / "framing" / "hand-of-god" / "sources.yaml",
    historical_yaml=REPO / "corpus" / "historical" / "hand-of-god" / "record.yaml",
    aggregate=aggregate,
    db_dir=REPO / "data" / "lance",
    # The fixture lands inside the web deploy root so Vercel (root dir: web/) bundles it.
    fixtures_dir=REPO / "web" / "fixtures",
    corpus_git_sha=corpus_sha,
)
# The bake always produces a fixture. If the derived SPLIT drifts from the documented
# Hand of God thesis, a "WARNING: derived SPLIT does not match..." line prints ABOVE —
# the fixture is still written so you can inspect exactly what was derived (next cell).
print("\nfixture written to", result.fixture_path)
print("evidence atoms in pool:", result.pool_size)

## 5 — Inspect the derived SPLIT

The payoff. The Hand of God stays debated **not because the Law is unclear** (`RULE_AMBIGUITY` ruled out) and **not because the act is unknowable** (`INDETERMINACY` ruled out — Maradona admitted it), but because of **what the referee could see in the moment** (`DECISION_TIME_DEFICIT`) and **which nation is watching** (`CULTURAL_PRIOR_BIAS`). Each cell carries the Guardian verdict that confirmed it.

In [ ]:
b = result.bundle
print(f"SETTLED ({b.settled_fact.status}): {b.settled_fact.statement}\n")
print("THE SPLIT —", b.split.headline, "\n")
for cell, seal in zip(b.split.cells, b.cell_seals):
    print(f"  {cell.axis:<22} {cell.state:<16} guardian={seal.seal.verdict}")
    print(f"      {cell.rationale}")
print("\nLENSES")
for sl in b.lenses:
    print(f"  {sl.output.lens:<11} {sl.output.stance:<22} guardian={sl.seal.verdict}")

In [ ]:
# The exact bytes the web app will read.
print(result.fixture_path.read_text(encoding="utf-8")[:1500], "...")

## 6 — Commit the fixture back to the repo

The fixture is the deliverable: it is committed to the repo and the web app reads it at build time. Set a token (Colab secret `GH_TOKEN`, or paste one with `repo` scope) and run the cell to push, **or** download the fixture and commit it locally.

The commit is authored as **Hriday Vig** to keep the history hand-authored.

In [ ]:
# Option A — download the fixture and commit it locally (no token needed).
from google.colab import files
files.download(str(result.fixture_path))

In [ ]:
# Option B — push directly. Provide a GitHub token with `repo` scope.
import os
GH_TOKEN = os.environ.get("GH_TOKEN", "")  # or set it here
if GH_TOKEN:
    !git -C /content/offside config user.name "Hriday Vig"
    !git -C /content/offside config user.email "vighriday@gmail.com"
    !git -C /content/offside add web/fixtures/hand-of-god-1986.json
    !git -C /content/offside commit -m "chore(fixtures): bake Hand of God incident bundle"
    push_url = REPO_URL.replace("https://", f"https://{GH_TOKEN}@")
    !git -C /content/offside push {push_url} HEAD:main
else:
    print("No GH_TOKEN set — use Option A (download) and commit locally.")